# Matched Tractor-Mix / SAIGE calibration QC

Ingests up to four result sets from the shared full-covariate-complete cohort:

1. Tractor-Mix + limited covariates
2. Tractor-Mix + full covariates
3. SAIGE + limited covariates
4. SAIGE + full covariates

Produces per-phenotype:

- Combined QQ plots and λGC summary
- Counts of tested variants and effective sample size (when available)
- Top-hit overlap (secondary to calibration)

**Comparability rule:** sample-count differences across models are flagged as
failures; do not interpret λGC differences when status=`FAIL`.

Also supports legacy single-run Tractor-Mix QC via `plot_tractor_results.py`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "compare_calibration.py",
    "plot_tractor_results.py",
)
from workspace_paths import data_root

ROOT = data_root()
COMPARE_PY = SCRIPTS / "compare_calibration.py"
PLOT_PY = SCRIPTS / "plot_tractor_results.py"

# Result directories (local). Leave unset / empty to skip a model.
TRACTOR_LIMITED_DIR = Path(os.environ.get("TRACTOR_LIMITED_DIR", "results/tractor_limited"))
TRACTOR_FULL_DIR = Path(os.environ.get("TRACTOR_FULL_DIR", "results/tractor_full"))
SAIGE_LIMITED_DIR = Path(os.environ.get("SAIGE_LIMITED_DIR", "results/saige_limited"))
SAIGE_FULL_DIR = Path(os.environ.get("SAIGE_FULL_DIR", "results/saige_full"))
NULL_META_DIR = Path(os.environ.get("SAIGE_NULL_META_DIR", "results/saige_null_meta"))

OUT_DIR = Path(os.environ.get("CALIBRATION_QC_DIR", "calibration_qc"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

ws = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")


def sh(cmd: str) -> None:
    print(cmd)
    rc = get_ipython().system(cmd)
    if rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


print("OUT_DIR:", OUT_DIR.resolve())
for label, d in [
    ("tractor_limited", TRACTOR_LIMITED_DIR),
    ("tractor_full", TRACTOR_FULL_DIR),
    ("saige_limited", SAIGE_LIMITED_DIR),
    ("saige_full", SAIGE_FULL_DIR),
]:
    n = len(list(d.glob("*.tsv"))) if d.exists() else 0
    print(f"  {label}: {d} ({n} tsv)")


In [ ]:
# Optional: pull each model's result TSVs from GCS globs
# Example:
#   export TRACTOR_LIMITED_GCS='gs://BUCKET/.../*.tractor_mix.tsv'
GCS_MAP = {
    TRACTOR_LIMITED_DIR: os.environ.get("TRACTOR_LIMITED_GCS", ""),
    TRACTOR_FULL_DIR: os.environ.get("TRACTOR_FULL_GCS", ""),
    SAIGE_LIMITED_DIR: os.environ.get("SAIGE_LIMITED_GCS", ""),
    SAIGE_FULL_DIR: os.environ.get("SAIGE_FULL_GCS", ""),
    NULL_META_DIR: os.environ.get("SAIGE_NULL_META_GCS", ""),
}

for dest, gcs in GCS_MAP.items():
    if not gcs:
        continue
    dest.mkdir(parents=True, exist_ok=True)
    sh(f"gsutil -m cp {gcs} {dest}/")

In [ ]:
cmd = [
    f"python3 {COMPARE_PY}",
    f"--out-dir {OUT_DIR}",
]
if TRACTOR_LIMITED_DIR.exists() and any(TRACTOR_LIMITED_DIR.glob("*.tsv")):
    cmd.append(f"--tractor-limited-dir {TRACTOR_LIMITED_DIR}")
if TRACTOR_FULL_DIR.exists() and any(TRACTOR_FULL_DIR.glob("*.tsv")):
    cmd.append(f"--tractor-full-dir {TRACTOR_FULL_DIR}")
if SAIGE_LIMITED_DIR.exists() and any(SAIGE_LIMITED_DIR.glob("*.tsv")):
    cmd.append(f"--saige-limited-dir {SAIGE_LIMITED_DIR}")
if SAIGE_FULL_DIR.exists() and any(SAIGE_FULL_DIR.glob("*.tsv")):
    cmd.append(f"--saige-full-dir {SAIGE_FULL_DIR}")
if NULL_META_DIR.exists() and any(NULL_META_DIR.glob("*.tsv")):
    cmd.append(f"--null-meta-dirs {NULL_META_DIR}")

assert any(
    x.startswith("--tractor-") or x.startswith("--saige-") for x in cmd
), "Provide at least one model result directory with *.tsv files"

sh(" ".join(cmd))

import pandas as pd
from IPython.display import Image, display, Markdown

summary = pd.read_csv(OUT_DIR / "calibration_summary.tsv", sep="\t")
flags = pd.read_csv(OUT_DIR / "comparability_flags.tsv", sep="\t")
display(Markdown("### Comparability flags"))
display(flags)
if (flags["status"] == "FAIL").any():
    print(
        "WARNING: sample-count mismatches detected. "
        "Treat λGC differences as non-comparable for those phenotypes."
    )
display(Markdown("### Calibration summary"))
display(summary)
display(Markdown((OUT_DIR / "calibration_summary.md").read_text()))

In [ ]:
# Per-phenotype combined QQ + overlap tables
for pheno_dir in sorted(p for p in OUT_DIR.iterdir() if p.is_dir()):
    print("===", pheno_dir.name, "===")
    fail = pheno_dir / "COMPARABILITY_FAIL.txt"
    if fail.exists():
        print(fail.read_text())
    qq = pheno_dir / "qq_matched.png"
    if qq.exists():
        display(Image(filename=str(qq)))
    ov = pheno_dir / "top_hit_overlap.tsv"
    if ov.exists():
        display(pd.read_csv(ov, sep="\t"))

## Optional: legacy single-run Tractor-Mix QC

Set `TRACTOR_RESULTS_DIR` to a directory of `*.tractor_mix.tsv` files to also
generate the original per-run QQ / Manhattan plots.

In [ ]:
LEGACY_DIR = Path(os.environ.get("TRACTOR_RESULTS_DIR", ""))
LEGACY_OUT = Path(os.environ.get("TRACTOR_QC_DIR", "tractor_mix_qc"))
if LEGACY_DIR and LEGACY_DIR.exists():
    legacy_files = sorted(LEGACY_DIR.glob("*.tractor_mix.tsv")) or sorted(LEGACY_DIR.glob("*.tsv"))
    if legacy_files:
        LEGACY_OUT.mkdir(exist_ok=True)
        files_arg = " ".join(str(p) for p in legacy_files)
        sh(f"python3 {PLOT_PY} --results {files_arg} --out-dir {LEGACY_OUT}")
        display(pd.read_csv(LEGACY_OUT / "pilot_summary.tsv", sep="\t"))
    else:
        print("No legacy Tractor result TSVs found")
else:
    print("Skipping legacy Tractor-only QC (set TRACTOR_RESULTS_DIR to enable)")

In [ ]:
if ws:
    dest = f"{ws}/tractor_mix_pilot/calibration_qc/"
    sh(f"gsutil -m cp -r {OUT_DIR}/* {dest}")
    print("Uploaded calibration QC to", dest)
else:
    print("WORKSPACE_BUCKET unset; QC left in", OUT_DIR.resolve())